# hw3_distillation

GPU T4х2, Internet включен


In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working/hw3_distillation")
TIME_BUDGET_HOURS = 10.5
PREVIOUS_RUN = None  # при продолжении: папка из прошлого Output с runs/ и data/


## Подготовка


In [ ]:
import os
import shutil

sources = [p.parent for p in Path("/kaggle/input").rglob("requirements.txt")
           if (p.parent / "src/cli.py").is_file()
           and (p.parent / "configs/teacher.json").is_file()
           and not (p.parent / "runs").exists()]
if len(sources) != 1:
    raise RuntimeError("подключите один Dataset с кодом проекта через Add Input")

ROOT.mkdir(parents=True, exist_ok=True)
for name in ("src", "configs"):
    shutil.copytree(sources[0] / name, ROOT / name, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("__pycache__", "*.pyc", ".DS_Store"))
shutil.copy2(sources[0] / "requirements.txt", ROOT / "requirements.txt")

if PREVIOUS_RUN:
    for name in ("runs", "data"):
        previous = Path(PREVIOUS_RUN) / name
        if not previous.is_dir():
            raise FileNotFoundError(f"не найдена папка {previous}")
        if not (ROOT / name).exists():
            shutil.copytree(previous, ROOT / name)

os.chdir(ROOT)
os.environ["HF_HOME"] = "/kaggle/temp/huggingface"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [ ]:
import subprocess
import sys

def run(*args):
    subprocess.run([sys.executable, "-u", *map(str, args)], cwd=ROOT, check=True)

run("-m", "pip", "install", "-q", "setuptools<81", "wheel")
run("-m", "pip", "install", "-q", "-r", "requirements.txt")
run("-c", "import torch; assert torch.cuda.is_available(), 'включите GPU в Settings'; "
    "print(torch.cuda.get_device_name(0))")


## Обучение и сжатие


In [ ]:
import json
import time
from tqdm.auto import tqdm

run("-m", "src.cli", "prepare")
gpu_count = int(subprocess.check_output(
    [sys.executable, "-c", "import torch; print(torch.cuda.device_count())"], text=True))
jobs = [
    ("teacher", set(), ["train", "teacher"]),
    ("cache", {"teacher"}, ["cache-teacher"]),
    ("student_kd", {"cache"}, ["train", "student_kd"]),
    ("factorized", {"teacher"}, ["train", "factorized"]),
    ("pruned_init", {"cache"}, ["train", "pruned_init"]),
    ("shared", {"student_kd"}, ["train", "shared"]),
    ("shared_lora", {"student_kd"}, ["train", "shared_lora"]),
    ("wanda", {"student_kd"}, ["prune", "wanda"]),
    ("sparsegpt", {"student_kd"}, ["prune", "sparsegpt"]),
]

def complete(name):
    path = ROOT / "runs" / name / "status.json"
    return path.exists() and json.loads(path.read_text())["status"] == "complete"

pending, active, done = jobs.copy(), {}, set()
deadline = time.monotonic() + TIME_BUDGET_HOURS * 3600
with tqdm(total=len(jobs), desc="этапы запуска", unit="этап") as progress:
    try:
        while pending or active:
            for gpu, (name, process) in list(active.items()):
                code = process.poll()
                if code is None:
                    continue
                if code != 0:
                    raise RuntimeError(f"ошибка на этапе {name}, код завершения {code}")
                if name != "cache" and not complete(name):
                    raise RuntimeError(f"этап {name} не завершён, сохранён последний шаг")
                done.add(name)
                del active[gpu]
                progress.update(1)
                tqdm.write(f"завершено: {name}")
            for job in pending.copy():
                name, dependencies, command = job
                if not dependencies <= done:
                    continue
                if name != "cache" and complete(name):
                    pending.remove(job)
                    done.add(name)
                    progress.update(1)
                    continue
                free = [gpu for gpu in range(min(2, gpu_count)) if gpu not in active]
                if not free:
                    continue
                remaining = (deadline - time.monotonic()) / 3600
                if remaining <= 0:
                    raise RuntimeError("время запуска закончилось, можно продолжить с сохранённых весов")
                args = command.copy()
                if args[0] == "train":
                    args += ["--time-budget-hours", str(remaining)]
                    if (ROOT / "runs" / name / "last.pt").exists():
                        args.append("--resume")
                gpu = free[0]
                # каждый процесс видит одну видеокарту и пишет в свою папку запуска
                env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu),
                           OMP_NUM_THREADS="2", MKL_NUM_THREADS="2")
                process = subprocess.Popen([sys.executable, "-u", "-m", "src.cli", *args],
                                           cwd=ROOT, env=env)
                active[gpu] = (name, process)
                pending.remove(job)
                tqdm.write(f"запуск: {name}, gpu {gpu}")
            if active:
                progress.set_postfix_str(", ".join(name for name, _ in active.values()))
                time.sleep(1)
            elif pending:
                raise RuntimeError("не удалось запустить оставшиеся этапы")
    finally:
        for _, process in active.values():
            if process.poll() is None:
                process.terminate()
        for _, process in active.values():
            process.wait()


## Результаты


In [ ]:
import json
import pandas as pd

names = ["teacher", "factorized", "student_kd", "pruned_init",
         "shared", "shared_lora", "wanda", "sparsegpt"]
unfinished = []
for name in names:
    path = ROOT / "runs" / name / "status.json"
    if not path.exists() or json.loads(path.read_text())["status"] != "complete":
        unfinished.append(name)

if not unfinished:
    run("-m", "src.cli", "evaluate", "all")
else:
    print("ещё не завершены: " + ", ".join(unfinished))

rows = []
for name in names:
    path = ROOT / "runs" / name / "test_metrics.json"
    if path.exists():
        metrics = json.loads(path.read_text())
        rows.append({"модель": name, "F1": metrics["f1"],
                     "параметры": metrics["parameters"],
                     "размер, МБ": metrics["checkpoint_mb"],
                     "время прохода, мс": metrics["benchmark"]["latency_ms"]})
if rows:
    display(pd.DataFrame(rows))
